# Tool-Using Agents with the Galileo Python SDK

This notebook keeps the agent, tool, workflow, and metric calls inline so you can see how Galileo logs a multi-step tool-using agent without a wrapper class.


In [2]:
import sys
from pathlib import Path

from dotenv import load_dotenv

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

print(f'Loaded credentials from {ENV_FILE}')


Loaded credentials from /Users/binliu/Documents/rungalileo/galileo-test/.env


In [3]:
from galileo import GalileoLogger, GalileoMetrics, Message, MessageRole, galileo_context
from galileo.config import GalileoPythonConfig
from galileo.decorator import log as galileo_log
from galileo.log_streams import create_log_stream, enable_metrics
from galileo.projects import delete_project

PROJECT_NAME = 'GalileoEval_S3_Agent'
LOG_STREAM_NAME = 'agent-monitoring'
MODEL = 'gpt-4o-mini'


## 1. Initialize Galileo

Initialize the Galileo context, make sure the log stream exists, and print the console links if Galileo returns IDs.


In [4]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

try:
    create_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)
except Exception:
    pass

config = GalileoPythonConfig.get()
logger = galileo_context.get_logger_instance()
project_id = getattr(logger, 'project_id', None)
log_stream_id = getattr(logger, 'log_stream_id', None)

if project_id and log_stream_id:
    project_url = f'{config.console_url}project/{project_id}'
    log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
    print(project_url)
    print(log_stream_url)
else:
    print('Galileo context initialized')


https://console.demo-v2.galileocloud.io/project/0feb40e7-316b-4bf7-86c1-e541d07f065f
https://console.demo-v2.galileocloud.io/project/0feb40e7-316b-4bf7-86c1-e541d07f065f/log-streams/8f24a48d-6400-4f42-aa67-ea2827a7475a


## 2. Log a Multi-Turn Agent Session


In [5]:
logger = GalileoLogger(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)
logger.start_session(name='travel-agent-session-001')

logger.start_trace(input='Plan me a 3-day trip to Tokyo.')
logger.add_agent_span(
    input='Plan me a 3-day trip to Tokyo.',
    output="I'll search for flights, hotels, and attractions in Tokyo.",
    name='TravelPlannerAgent',
    duration_ns=100_000_000,
)
logger.add_llm_span(
    input=[Message(role=MessageRole.user, content='Plan me a 3-day trip to Tokyo.')],
    output=Message(role=MessageRole.assistant, content="I'll help plan your Tokyo trip. Let me search for options."),
    model=MODEL,
)
logger.conclude(output="I'll help plan your Tokyo trip. Let me search for options.")

logger.start_trace(input='Find flights from SFO to Tokyo next month.')
logger.add_tool_span(
    input='search_flights(origin="SFO", destination="NRT", date="2026-04-10")',
    output='[{"airline":"ANA","price":850},{"airline":"JAL","price":920}]',
    duration_ns=300_000_000,
)
logger.add_tool_span(
    input='search_hotels(city="Tokyo", checkin="2026-04-10", nights=3)',
    output='[{"name":"Shinjuku Granbell","price_per_night":120},{"name":"Park Hyatt","price_per_night":450}]',
    duration_ns=250_000_000,
)
logger.add_llm_span(
    input=[Message(role=MessageRole.user, content='Find flights and hotels for Tokyo trip.')],
    output=Message(role=MessageRole.assistant, content='ANA at $850 and Shinjuku Granbell at $120/night are the best value.'),
    model=MODEL,
)
logger.conclude(output='Found flights and hotels for Tokyo trip.')

logger.start_trace(input='Book the ANA flight and Shinjuku Granbell hotel.')
logger.add_tool_span(
    input='book_flight(airline="ANA", flight_id="NH107", passengers=1)',
    output='{"confirmation":"ANA-2026-XK9F","status":"confirmed"}',
    duration_ns=500_000_000,
)
logger.add_tool_span(
    input='book_hotel(hotel_id="shinjuku-granbell", checkin="2026-04-10", nights=3)',
    output='{"confirmation":"SG-88721","status":"confirmed"}',
    duration_ns=400_000_000,
)
logger.add_llm_span(
    input=[Message(role=MessageRole.user, content='Book the ANA flight and Shinjuku Granbell.')],
    output=Message(role=MessageRole.assistant, content='Both booked. Flight ANA-2026-XK9F and hotel SG-88721 confirmed.'),
    model=MODEL,
)
logger.conclude(output='Trip booked successfully.')
logger.flush()
'3-turn agent session with tool calls'


'3-turn agent session with tool calls'

## 3. Log a Nested Workflow


In [6]:
logger = GalileoLogger(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)
logger.start_trace(input='Generate a complete travel itinerary.')
logger.add_workflow_span(
    input='Generate itinerary for Tokyo trip',
    output='3-day itinerary generated with attractions, restaurants, and transport.',
    name='ItineraryWorkflow',
    duration_ns=800_000_000,
)
logger.add_agent_span(
    input='Research top attractions in Tokyo',
    output='Found Senso-ji, Shibuya Crossing, Meiji Shrine, and Tsukiji Market.',
    name='ResearchAgent',
)
logger.add_tool_span(
    input='search_attractions(city="Tokyo", category="must-see")',
    output='["Senso-ji Temple","Shibuya Crossing","Meiji Shrine","Tsukiji Outer Market"]',
)
logger.add_llm_span(
    input=[Message(role=MessageRole.user, content='Create a day-by-day itinerary.')],
    output=Message(role=MessageRole.assistant, content='Day 1: Asakusa. Day 2: Shibuya. Day 3: Meiji Shrine and Shinjuku.'),
    model=MODEL,
)
logger.conclude(output='Itinerary complete.')
logger.flush()
'Workflow trace logged'


'Workflow trace logged'

## 4. Log a Decorated Tool Flow


In [7]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

@galileo_log(span_type='tool')
def search_restaurants(city: str, cuisine: str) -> str:
    return f'[{{"name":"Sukiyabashi Jiro","city":"{city}","cuisine":"{cuisine}"}}]'

@galileo_log(span_type='agent')
def food_agent(query: str) -> str:
    return f"Found restaurants: {search_restaurants(city='Tokyo', cuisine='sushi')}"

@galileo_log(span_type='workflow')
def plan_dinner(user_request: str) -> str:
    return f'Dinner plan: {food_agent(user_request)}'

result = plan_dinner('Find the best sushi in Tokyo')
galileo_context.flush()
result


'Dinner plan: Found restaurants: [{"name":"Sukiyabashi Jiro","city":"Tokyo","cuisine":"sushi"}]'

## 5. Enable Agentic Metrics


In [8]:
enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[
        GalileoMetrics.action_advancement_luna,
        GalileoMetrics.agent_efficiency,
        GalileoMetrics.agent_flow,
    ],
)


[]

## 6. Enable Tool Metrics


In [9]:
enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[GalileoMetrics.tool_error_rate, GalileoMetrics.tool_selection_quality],
)


[]

## 7. Enable Reasoning and Session Metrics


In [10]:
enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[
        GalileoMetrics.reasoning_coherence,
        GalileoMetrics.action_completion_luna,
        GalileoMetrics.conversation_quality,
        GalileoMetrics.user_intent_change,
    ],
)


[]

## 8. Clean Up the Demo Project


In [ ]:
delete_project(name=PROJECT_NAME)
